# 01 · Data audit — does our data reproduce the official record?

**CHECKPOINT 1.** Before drawing a single chart we check that the pipeline's numbers match an independent
official source, list what data exists for which seasons, and fix the **data cut-off**.

Tables come from `python -m src.results` (Jolpica-F1, cached). Official figures come from
`data/reference/official_totals.csv` (source + date inside the file).

In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd
from src import config, stats
pd.set_option('display.width', 140)
P = config.PROCESSED
r   = pd.read_parquet(P/'results.parquet')
fr  = pd.read_parquet(P/'field_results.parquet')
fq  = pd.read_parquet(P/'field_qualifying.parquet')
st  = pd.read_parquet(P/'standings.parquet')
off = pd.read_parquet(P/'official_standings.parquet')
wm  = pd.read_parquet(P/'win_margins.parquet')
gp  = r[r.session == 'gp']

## 1 · Data cut-off
The latest Grand Prix in the data. Every page and README must show this line.

In [2]:
last = gp.sort_values('date').iloc[-1]
print(f"Data cut-off: {last.season} {last.race_name} (round {last['round']}, {last.date.date()})")
print('Jolpica manifest:', json.loads(config.MANIFEST.read_text())['fetched_at'])

Data cut-off: 2026 Azerbaijan Grand Prix (round 15, 2026-09-26)
Jolpica manifest: 2026-09-29T13:16:14+00:00


## 2 · Career totals — ours vs official

In [3]:
official = pd.read_csv(config.REFERENCE/'official_totals.csv', comment='#').set_index('metric')['value']
c = stats.gp_counts(r)
fastest_q = int(((fq.driver_id == config.DRIVER_ID) & (fq.quali_pos == 1)).sum())
titles = int(((off.driver_id == config.DRIVER_ID) & (off.position == 1) & (off.season < last.season)).sum())
ours = pd.Series({'starts': c['starts'], 'wins': c['wins'], 'podiums': c['podiums'], 'poles': fastest_q,
                  'career_points': r.points.sum(), 'championships': titles})
cmp = pd.DataFrame({'ours': ours, 'official': official.reindex(ours.index)})
cmp['match'] = (cmp.ours - cmp.official).abs() < 0.01
cmp

,ours,official,match
starts,248.0,248.0,True
wins,71.0,71.0,True
podiums,134.0,134.0,True
poles,51.0,48.0,False
career_points,3607.5,3607.5,True
championships,4.0,4.0,True


Fastest laps are not audited: we don't use them in any chart.

Championships count only completed seasons (the live season's leader is not a champion).

## 3 · Pole reconciliation

We count **fastest qualifier** = Jolpica qualifying P1. It differs from the official pole count because:
* in **2021 sprint weekends** pole was awarded to the sprint winner, not the fastest qualifier;
* after qualifying, times can be **deleted** or **grid penalties** applied, and sources differ on whether that
  driver keeps the pole in the record books.

The tables below list every race where 'fastest qualifier' and 'started P1' disagree.

In [4]:
q_me = fq[fq.driver_id == config.DRIVER_ID][['season','round','race_name','quali_pos']]
g = q_me.merge(gp[['season','round','grid']], on=['season','round'])
print('Fastest qualifier but did NOT start P1:'); display(g[(g.quali_pos == 1) & (g.grid != 1)])
print('Started P1 but was NOT fastest qualifier:'); display(g[(g.quali_pos != 1) & (g.grid == 1)])

Fastest qualifier but did NOT start P1:


,season,round,race_name,quali_pos,grid
98,2019,18,Mexican Grand Prix,1,4
154,2022,14,Belgian Grand Prix,1,14
174,2023,12,Belgian Grand Prix,1,6
198,2024,14,Belgian Grand Prix,1,11
207,2024,23,Qatar Grand Prix,1,2


Started P1 but was NOT fastest qualifier:


,season,round,race_name,quali_pos,grid
128,2021,10,British Grand Prix,2,1
132,2021,14,Italian Grand Prix,3,1


**Reading:** ours − official = 3. Seven races are candidates: five where he was fastest but did not start P1 (time deleted or grid penalty) and two 2021 sprint weekends where the starting spot came from the sprint. Which of them the record books credit to him depends on the record-keeper's convention, so the exact reconciliation is **to be confirmed with a second source before any pole number appears in prose.** Until then the case study uses the unambiguous *fastest qualifier* metric, defined in the methodology.

## 4 · Per-season record

In [5]:
sp = r[r.session == 'sprint']
fq_me = fq[(fq.driver_id == config.DRIVER_ID) & (fq.quali_pos == 1)]
season = gp.groupby('season').agg(
    team=('constructor', lambda s: ' / '.join(dict.fromkeys(s))),
    number=('number', lambda s: ' / '.join(map(str, dict.fromkeys(s)))),
    starts=('status_cat', lambda s: (s != 'DNS').sum()),
    wins=('position_text', lambda s: (s == '1').sum()),
    podiums=('position_text', lambda s: s.isin(['1','2','3']).sum()),
    not_classified=('classified', lambda s: (~s).sum()),
)
season['fastest_qualifier'] = fq_me.groupby('season').size().reindex(season.index, fill_value=0)
season['sprints'] = sp.groupby('season').size().reindex(season.index, fill_value=0)
season['points'] = gp.groupby('season').points.sum() + sp.groupby('season').points.sum().reindex(season.index, fill_value=0)
fin = off[off.driver_id == config.DRIVER_ID].set_index('season')
season['champ_pos'] = fin.position; season['after_round'] = fin.after_round
season

,team,number,starts,wins,podiums,not_classified,fastest_qualifier,sprints,points,champ_pos,after_round
season,,,,,,,,,,,
2015,Toro Rosso,33,19,0,0,4,0,0,49.0,12,19
2016,Toro Rosso / Red Bull,33,21,1,7,3,0,0,204.0,5,21
2017,Red Bull,33,20,2,4,7,0,0,168.0,6,20
2018,Red Bull,33,21,2,11,4,0,0,249.0,4,21
2019,Red Bull,33,21,3,9,2,3,0,278.0,3,21
2020,Red Bull,33,17,2,11,5,1,0,214.0,3,17
2021,Red Bull,33,22,10,18,3,8,3,395.5,1,22
2022,Red Bull,1,22,15,17,1,8,3,454.0,1,22
2023,Red Bull,1,22,19,21,0,13,6,575.0,1,22


## 5 · Standings reconstruction
`standings.parquet` rebuilds cumulative points round by round from race + sprint points. Check the last round of each season against the official Jolpica standings.

In [6]:
lastr = st.sort_values('round').groupby(['season','driver_id']).tail(1)
m = off.merge(lastr, on=['season','driver_id'])
print('driver-seasons compared:', len(m), '| points mismatches:', int(((m.points - m.points_cum).abs() > 0.01).sum()))
tied = m.duplicated(['season','points'], keep=False)   # tie partners searched across the full table
mm = m[m.position != m['rank']]
print('rank mismatches:', len(mm), '— all ties on points or officially unranked (position 0):',
      bool((tied[mm.index] | (mm.position == 0)).all()))
print('mismatches in top 3:', int(((mm.position.between(1,3)) | (mm['rank'] <= 3)).sum()))

driver-seasons compared: 267 | points mismatches: 0
rank mismatches: 41 — all ties on points or officially unranked (position 0): True
mismatches in top 3: 0


Rank ties are broken officially by count-back (most wins, then most 2nds…), which we don't reproduce. It never affects the top 3, and C02 plots points, not ranks.

## 6 · How races ended — status categories

In [7]:
pd.crosstab(gp.season, gp.status_cat)

status_cat,DNF-incident,DNF-mechanical,DNF-unknown,Finished,Lapped
season,,,,,
2015,2,3,0,8,6
2016,1,2,0,18,0
2017,3,4,0,12,1
2018,2,2,0,17,0
2019,1,1,0,19,0
2020,3,2,0,12,0
2021,3,0,0,19,0
2022,0,2,0,20,0
2023,0,0,0,22,0


⚠️ **From 2024 Jolpica reports every retirement as a generic `Retired`** with no cause, so these become
`DNF-unknown`. C13 (DNF anatomy) must show 2024+ as 'cause not recorded' rather than guess.

Categories are defined in `data/reference/status_map.csv`; rows marked REVIEW need a human decision.

In [8]:
smap = pd.read_csv(config.REFERENCE/'status_map.csv', comment='#')
smap[smap.note.fillna('').str.startswith('REVIEW')]

,status,category,note
55,Puncture,DNF-incident,"REVIEW: punctures are usually debris/contact, ..."
56,Tyre,DNF-mechanical,REVIEW: tyre failure treated as equipment failure
57,Front wing,DNF-incident,REVIEW: front-wing DNFs are almost always cont...
58,Undertray,DNF-mechanical,REVIEW: could be damage
59,Out of fuel,DNF-other,"REVIEW: team/fuel-calculation error, neither m..."


## 7 · Coverage by source

In [9]:
cov = fr.groupby(['season','session']).agg(rounds=('round','nunique')).unstack(fill_value=0)
cov.columns = [f'{b}_rounds' for _, b in cov.columns]
cov['max_gp_entries'] = gp.groupby('season').size()
cov['quali_rounds'] = fq.groupby('season')['round'].nunique()
cov['fastf1_timing'] = ['yes' if s >= 2018 else 'no (pre-2018)' for s in cov.index]
cov

,gp_rounds,sprint_rounds,max_gp_entries,quali_rounds,fastf1_timing
season,,,,,
2015,19,0,19,19,no (pre-2018)
2016,21,0,21,21,no (pre-2018)
2017,20,0,20,20,no (pre-2018)
2018,21,0,21,21,yes
2019,21,0,21,21,yes
2020,17,0,17,17,yes
2021,22,3,22,22,yes
2022,22,3,22,22,yes
2023,22,6,22,22,yes


* **Jolpica** covers every round of every season 2015 → cut-off: results, sprints, qualifying, standings.
* **FastF1 live timing (laps, stints, telemetry, car position) exists only from 2018.** 2015–2017 charts use
  results and lap times only.
* Per the light-scope decision (CLAUDE.md §0.1) we do **not** bulk-download FastF1. It is loaded only for the few
  sessions C05, C06 and C08 need; availability is checked when those sessions are chosen.
* Win margins (C07): races where P2 finished a lap down have no time gap and are excluded:

In [10]:
print(f"{wm.margin_s.notna().sum()} of {len(wm)} wins have a measurable gap to P2")
wm[wm.margin_s.isna()]

71 of 71 wins have a measurable gap to P2


,season,round,date,race_name,driver_id_p2,margin_s,margin_pct


## 8 · Sanity check — a known result

In [11]:
fw = gp[gp.position_text == '1'].iloc[0]
print(f"First GP win: {fw.season} {fw.race_name}, grid {fw.grid} → P{fw.position}, {fw.constructor}, car #{fw.number}")
d = gp.iloc[0]
print(f"Debut: {d.season} {d.race_name}, {d.constructor}, car #{d.number}")
print('Car-number eras (verified by src.results.verify_config):', gp.groupby('number').season.agg(['min','max']).to_dict('index'))

First GP win: 2016 Spanish Grand Prix, grid 4 → P1, Red Bull, car #33
Debut: 2015 Australian Grand Prix, Toro Rosso, car #33
Car-number eras (verified by src.results.verify_config): {1: {'min': 2022, 'max': 2025}, 3: {'min': 2026, 'max': 2026}, 33: {'min': 2015, 'max': 2021}}
